In [1]:
from sedona.spark import *

In [2]:
config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Setting Spark log level to "WARN".
26/03/12 13:12:11 INFO core/src/lib.rs: Sedona native acceleration engine v0.12.0 ready
                                                                                

In [3]:
database = 'gde_silver'
sedona.sql(f'CREATE DATABASE IF NOT EXISTS org_catalog.{database}')

DataFrame[]

In [4]:
prefix = 's3://wherobots-examples/gdea-course-data/raw-data/'

In [5]:
homes = sedona.read.format('csv') \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .option('skipRows', 1) \
    .load(f'{prefix}' + 'sales_king_county.csv')

In [6]:
homes.show()

+---+----------+------------+----------+----------+--------+------------+-----------+---------+----------------+-----------------+----+-------------+-------+--------------------+-----------+--------+-------+----------+---------+--------+----+------+----------+-----+-----------+---------+-------+----+---------+---------+---------+---------+---------+----+----+---------+-------------+------------+-------------+-------------+----------------+------------+----------+-------------+-------------+---------------+----------+---------+
|_c0|   sale_id|        pinx| sale_date|sale_price|sale_nbr|sale_warning|join_status|join_year|        latitude|        longitude|area|         city| zoning|         subdivision|present_use|land_val|imp_val|year_built|year_reno|sqft_lot|sqft|sqft_1|sqft_fbsmt|grade|fbsmt_grade|condition|stories|beds|bath_full|bath_3qtr|bath_half|garb_sqft|gara_sqft|wfnt|golf|greenbelt|noise_traffic|view_rainier|view_olympics|view_cascades|view_territorial|view_skyline|view_sound|

In [7]:
homes.writeTo(f"org_catalog.gde_bronze.king_co_homes").createOrReplace()

In [8]:
sedona.sql(f'''
alter table org_catalog.gde_bronze.king_co_homes add column geometry geometry
''')

DataFrame[]

In [9]:
sedona.sql(f'''
update org_catalog.gde_bronze.king_co_homes
set geometry = st_point(longitude, latitude) 
''')

DataFrame[]

In [10]:
import wkls

washington = wkls.us.wa.wkt()
seattle = wkls.us.wa.seattle.wkt()
kirkland = wkls.us.wa.kirkland.wkt()
bellevue = wkls.us.wa.bellevue.wkt()

# Small bounding box in central Kirkland (~0.5km²)
kirkland_small = "POLYGON((-122.212 47.676, -122.200 47.676, -122.200 47.682, -122.212 47.682, -122.212 47.676))"

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
# 26/03/01 20:26:18 WARN JoinQueryDetector: Filter pushdown detected on the object side of a KNN join. 
# This may cause the KNN join to return incorrect results. Consider materializing the object side before 
# the join to prevent filter pushdown.

# Pre-filter buildings into a temp
sedona.sql(f'''
create or replace temp view kirkland_buildings as
select * from wherobots_open_data.overture_maps_foundation.buildings_building
where st_intersects(geometry, ST_GeomFromWKT('{kirkland_small}'))
''')

# Now run KNN against bounded map
sedona.sql(f'''
create or replace table org_catalog.gde_bronze.king_co_homes_conflated as
    select a.*,
        a.geometry as point,
        b.geometry as polygon,
        b.id as overture_id,
        b.height
    from org_catalog.gde_bronze.king_co_homes a
    join kirkland_buildings b
        on st_knn(a.geometry, b.geometry, 1, true, 100)
    where st_intersects(a.geometry, ST_GeomFromWKT('{kirkland_small}'))
''')

# sedona.sql(f'''
# create or replace table org_catalog.gde_bronze.king_co_homes_conflated as
#     select a.*,
#         a.geometry as point,
#         b.geometry as polygon,
#         b.id as overture_id,
#         b.height
#     from org_catalog.gde_bronze.king_co_homes a
#     join wherobots_open_data.overture_maps_foundation.buildings_building b
#         on st_knn(a.geometry, b.geometry, 1, true, 100)
#     where 
#         st_intersects(a.geometry, ST_GeomFromWKT('{kirkland_small}')) and 
#         st_intersects(b.geometry, ST_GeomFromWKT('{kirkland_small}'))
# ''')

26/03/12 13:12:58 WARN JoinQueryDetector: Filter pushdown detected on the object side of a KNN join. This may cause the KNN join to return incorrect results. Consider materializing the object side before the join to prevent filter pushdown.
                                                                                

DataFrame[]

In [12]:
sedona.sql("SHOW TABLES IN org_catalog.gde_silver").show()

+----------+--------------------+-----------+
| namespace|           tableName|isTemporary|
+----------+--------------------+-----------+
|gde_silver|         census_data|      false|
|gde_silver|       homes_buffers|      false|
|gde_silver|  homes_demographics|      false|
|gde_silver|homes_distance_to...|      false|
|gde_silver|homes_distance_to...|      false|
|gde_silver| homes_flood_hazards|      false|
|gde_silver| homes_school_scores|      false|
|gde_silver|homes_seismic_haz...|      false|
|gde_silver|homes_zoning_over...|      false|
|gde_silver|house_sales_ndvi_...|      false|
|gde_silver|house_sales_tri_s...|      false|
|gde_silver|house_sales_zonal...|      false|
|gde_silver|residential_build...|      false|
|gde_silver|residential_build...|      false|
|gde_silver|     roads_proximity|      false|
|gde_silver|         school_data|      false|
|gde_silver|     zoning_overlaps|      false|
+----------+--------------------+-----------+

